1. Anatomía del GIL

El cálculo matemático no mejora mucho usando threading porque Python tiene el GIL, que hace que un solo hilo pueda ejecutar un solo codigo a la vez. Por eso, aunque haya varios hilos, no se aprovechan realmente varios núcleos para tareas pesadas de CPU. En cambio, con multiprocessing cada proceso tiene su propio intérprete y puede ejecutarse en distintos núcleos, logrando un mejor rendimiento.

2. Context Switching

En asyncio el cambio entre tareas es más liviano porque ocurre dentro del mismo hilo y se realiza cuando una tarea usa await. En threading el sistema operativo tiene que administrar los hilos y cambiar entre ellos, lo que genera un costo mayor de recursos.

3. Serialización e IPC

Cuando se usan procesos distintos, los datos tienen que enviarse de un proceso a otro. Para hacerlo, Python utiliza Pickling, que convierte los objetos en datos que pueden transferirse. Si los objetos son muy grandes, este proceso puede tardar más tiempo y afectar el rendimiento general de la aplicación.

4. Thread-Safety

queue.Queue está diseñada para que varios hilos puedan usarla de forma segura al mismo tiempo. Si se utilizara una lista común, podrían aparecer problemas cuando varios hilos intentan agregar o quitar elementos simultáneamente, produciendo resultados incorrectos.

In [ ]:
import asyncio
import threading
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import queue
import time
import os
import random

vehiculos_procesados = 0
lock_bd = threading.Lock()
cola_escritura = queue.Queue(maxsize=5000)


def worker_db_legacy():
    global vehiculos_procesados

    while True:
        try:
            registro = cola_escritura.get()

            if registro is None:
                break
            time.sleep(0.02)

            with lock_bd:
                vehiculos_procesados += 1
            cola_escritura.task_done()
        except Exception as e:
            break


def recalculo_pesado_rutas(chunk_datos):
    resultados = []

    for dato in chunk_datos:

        total = 0
        for i in range(1000000):
            total += i ** 2

        dato['estado_ruta'] = "Aprobada"
        dato['procesado_por_pid'] = os.getpid()
        
        resultados.append(dato)
    return resultados


async def peticion_telemetria_sensor(sensor_id):

    espera_red = random.uniform(0.1, 0.8)
    await asyncio.sleep(espera_red)

    return {
        "vehiculo_id": f"TRK-{sensor_id}",
        "latitud": random.uniform(-90, 90),
        "longitud": random.uniform(-180, 180),
        "velocidad_kmh": round(random.uniform(40, 120), 2),
        "timestamp": time.time()
    }


async def orquestador_principal():

    print(f"[*] iniciando orquestador (PID: {os.getpid()})...")

    for _ in range(4):
        hilo = threading.Thread(target=worker_db_legacy, daemon=True)
        hilo.start()

    cores = os.cpu_count() or 4

    with ProcessPoolExecutor(max_workers=cores) as executor:

        tareas_red = [
            peticion_telemetria_sensor(i)
            for i in range(100)
        ]

        print("[*] esperando telemetria de la flota...")

        for tarea in asyncio.as_completed(tareas_red):

            dato = await tarea
            futuro = executor.submit(
                recalculo_pesado_rutas,
                [dato]
            )
            resultado = futuro.result()

            for registro in resultado:
                cola_escritura.put(registro)

    print("[*] flota procesada. sincronizando base de datos...")

    cola_escritura.join()

    print("[!] proceso finalizado.")

inicio = time.perf_counter()

await orquestador_principal()

print(f"\nreporte final")
print(f"tiempo total: {time.perf_counter() - inicio:.2f}s")
print(f"vehiculos guardados en BD: {vehiculos_procesados}")